# 04 · Evaluate: the chosen model, the slices, the decision, one look at July
**Owner: P3** · Status: `[ ]`

Order is the protocol: (1) ladder on June, (2) choose the buffer on June, (3) **freeze**, (4) July once, (5) slices, (6) interpretation. Everything before §4 uses May (fit) and June (score) only.

`RUN_JULY` must be switched on deliberately for the single final run. `lock_test()` then refuses any second run.

In [ ]:
import sys; sys.path.insert(0, "..")          # repo root, so `import src...` works
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.config import TARGET
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 140)

from src.clean import build_datasets
from src.config import FINAL_FEATURE_SET, FINAL_MODEL, FINAL_PARAMS, BUFFER_Q, BUFFER_RULE
from src.features import add_features
from src.models import make_pipeline, make_model, MODELS, MedianBaseline, RateRule, TLC_START, TLC_PER_MILE
from src.evaluate import (metrics, bootstrap_vs_baseline, slice_table, fit_quote_rule, apply_quote_rule,
                          quote_report, perturb_distance, save_results, lock_test, LOCK)

RUN_JULY = False                                       # flip to True ONLY for the single final run

D = build_datasets()
train, val, test = D["train"], D["val"], D["test"]
ytr, yva = train[TARGET], val[TARGET]
print("final:", FINAL_FEATURE_SET, FINAL_MODEL, FINAL_PARAMS, "| buffer:", BUFFER_RULE, BUFFER_Q)

## 1. The baseline ladder and the models on June (trained on May)

In [ ]:
fitted = {
    "1 median (May)": MedianBaseline().fit(train, ytr),
    "2a rate rule (TLC $3 + $3.50/mi)": RateRule(start=TLC_START, per_mile=TLC_PER_MILE).fit(train, ytr),
    "2b rate rule (fit on May)": RateRule().fit(train, ytr),
    "3 tree depth 2": make_pipeline(make_model("tree2"), "f4_shape").fit(train, ytr),
    "skeleton: linear f4_shape": make_pipeline(make_model("linear"), "f4_shape").fit(train, ytr),
    "FINAL: hgb f7_time_x": make_pipeline(make_model(FINAL_MODEL, **FINAL_PARAMS), FINAL_FEATURE_SET).fit(train, ytr),
}
BEST, BASE, RULE = "FINAL: hgb f7_time_x", "1 median (May)", "2b rate rule (fit on May)"
pred_val = {k: m.predict(val) for k, m in fitted.items()}
june_tab = pd.DataFrame({k: metrics(yva, p) for k, p in pred_val.items()}).T.drop(columns="n").round(3)
june_tab

FILL_LADDER

### 1b · Bias and the error distribution (June)
`bias` = mean(actual − predicted): negative = over-prediction.

In [ ]:
p = pred_val[BEST]; res = yva.to_numpy() - p
vs = add_features(val)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].hist(np.clip(res, -20, 30), bins=120); ax[0].axvline(0, color="k", lw=.8)
ax[0].set(title="June residuals (actual - predicted), clipped to [-20, 30]", xlabel="$")
dec = pd.qcut(p, 10)
cal = pd.DataFrame({"pred": p, "y": yva.to_numpy()}).groupby(dec, observed=True).agg(mean_pred=("pred", "mean"), mean_actual=("y", "mean"), median_actual=("y", "median"))
ax[1].plot(cal.mean_pred, cal.mean_actual, "o-", label="mean actual"); ax[1].plot(cal.mean_pred, cal.median_actual, "s-", label="median actual")
ax[1].plot(cal.mean_pred, cal.mean_pred, "k--", lw=.8, label="y = x"); ax[1].legend(); ax[1].set(title="calibration by decile of prediction", xlabel="mean predicted $")
pd.Series(res).groupby(vs.hour.to_numpy()).mean().plot(ax=ax[2], marker="o", title="mean residual by pickup hour (June)"); ax[2].axhline(0, color="k", lw=.8)
plt.tight_layout()
print("residual quantiles:", np.quantile(res, [.05, .25, .5, .75, .9, .95]).round(2), "| share |res| > $10:", round(float((np.abs(res) > 10).mean()), 4))
for by in ["dist_band", "hour_band", "airport_trip", "pickup_borough"]:
    display(slice_table(vs, yva, p, by).round(3))

FILL_BIAS

## 2. The decision: how large a buffer for 90% coverage? (chosen on June)
Quote = rule(prediction); we want actual ≤ quote on **90%** of trips. Three rules:
- **additive**: quote = p + b (one global b)
- **multiplicative**: quote = max(p, \$3) × k
- **by_dist_band**: quote = p + b[distance band] (one b per band; bands with < 200 June trips fall back to the global b)

**Selection criterion, fixed before this cell ran:** a rule fitted and scored on the same June trips hits 90% by construction, so that comparison would always favour the more flexible rule. We therefore **cross-fit within June**: fit on odd days, score on even days, and vice versa. We pick the rule with the smallest *worst-slice* coverage gap |coverage − 0.90| over distance band, hour band, airport flag and pickup borough (slices with n ≥ 1 000). Ties within 0.01 go to the lower mean cushion (quote − actual).

In [ ]:
y = yva.to_numpy(); band = vs.dist_band.astype(str).to_numpy(); day = val.tpep_pickup_datetime.dt.day.to_numpy()
rows, xq = [], {}
for kind in ["additive", "multiplicative", "by_dist_band"]:
    q = np.empty(len(y))
    for h in (0, 1):
        fit, ev = day % 2 == h, day % 2 != h
        q[ev] = apply_quote_rule(fit_quote_rule(kind, y[fit], p[fit], BUFFER_Q, slices=band[fit]), p[ev], band[ev])
    xq[kind] = q
    gaps = {by: float((slice_table(vs, y, p, by, quote=q, min_n=1000).coverage - BUFFER_Q).abs().max())
            for by in ["dist_band", "hour_band", "airport_trip", "pickup_borough"]}
    rows.append(dict(rule=kind, **quote_report(y, q), worst_gap=max(gaps.values()), **{f"gap_{k}": v for k, v in gaps.items()}))
rules_cmp = pd.DataFrame(rows).set_index("rule").round(3)
display(rules_cmp)
best_rule = rules_cmp.worst_gap.idxmin()
print("selected by the criterion:", best_rule)
assert best_rule == BUFFER_RULE, "config.BUFFER_RULE does not match the June criterion"

In [ ]:
# Final rule: refit on ALL of June, then frozen.
rule = fit_quote_rule(BUFFER_RULE, y, p, BUFFER_Q, slices=band)
print(rule)
quote_va = apply_quote_rule(rule, p, band)
print({k: round(v, 3) for k, v in quote_report(y, quote_va).items()})
for by in ["dist_band", "hour_band", "airport_trip", "pickup_borough"]:
    display(slice_table(vs, y, p, by, quote=xq[BUFFER_RULE]).round(3).rename(columns={"coverage": "coverage (cross-fit)"}))

FILL_RULE

### 2b · How much does the planned-distance assumption cost? (June)
The model stays as trained; only the distance it is *given* is perturbed with mean-one log-normal noise (seeded), mimicking a routing estimate that is off by ~5/10/20%.

In [ ]:
pipe = fitted[BEST]
sens = []
for sd in [0, 0.05, 0.10, 0.20]:
    pn = pipe.predict(perturb_distance(val, sd))
    sens.append(dict(noise_sd=sd, **{k: round(v, 3) for k, v in metrics(y, pn).items() if k != "n"},
                     coverage=round(quote_report(y, apply_quote_rule(rule, pn, band))["coverage"], 3)))
sens = pd.DataFrame(sens); sens

FILL_SENS

## 3. FREEZE: model, features, buffer rule (Gate 3)
FILL_FREEZE3

In [ ]:
save_results({"model": FINAL_MODEL, "feature_set": FINAL_FEATURE_SET, "params": FINAL_PARAMS, "quote_rule": rule,
              "june_ladder": june_tab.to_dict(orient="index"), "june_quote": quote_report(y, quote_va),
              "june_rule_comparison": rules_cmp.to_dict(orient="index"), "june_distance_noise": sens.to_dict(orient="records")},
             "gate3_freeze.json")

## 4. July: scored once
> **Disclosure.** July was already scored once, on **2026-11-01**, by the skeleton version of this notebook (linear model, f4_shape, additive buffer, *before* any model or buffer was frozen). That run reported July MAE 2.512 and 92.4% coverage; it is preserved in the git history (commit `e5b2824`). No decision in NB03 or §1–3 above used July; all were made on June.

`lock_test()` raises if July was already scored. Run only with `RUN_JULY = True`, after §3.

In [ ]:
if RUN_JULY:
    lock_test()                                           # one look at July
    yte = test[TARGET].to_numpy()
    ts = add_features(test)
    band_te = ts.dist_band.astype(str).to_numpy()
    pred_te = {k: m.predict(test) for k, m in fitted.items()}
    final_te = pred_te[BEST]
    quote_te = apply_quote_rule(rule, final_te, band_te)
    days = test.tpep_pickup_datetime.dt.date              # resample whole days (trips within a day are correlated)
    july_tab = pd.DataFrame({k: metrics(yte, pp) for k, pp in pred_te.items()}).T.round(3)
    vs_median = bootstrap_vs_baseline(yte, final_te, pred_te[BASE], groups=days, n_boot=2000)
    vs_rule = bootstrap_vs_baseline(yte, final_te, pred_te[RULE], groups=days, n_boot=2000)
    july_quote = quote_report(yte, quote_te)
    display(july_tab); display(vs_median.round(3)); display(vs_rule.round(3))
    print({k: round(v, 3) for k, v in july_quote.items()})
else:
    print("RUN_JULY is False: July not touched.")

## 5. Slices (July, with n): where does it fail?

In [ ]:
if RUN_JULY:
    slices = {}
    for by in ["pickup_borough", "airport_trip", "hour_band", "dist_band"]:
        slices[by] = slice_table(ts, yte, final_te, by, quote=quote_te).round(3)
        print(f"--- {by}"); display(slices[by])
    # Rush hour in Manhattan (weekday 16-19)
    ts["rush_manhattan"] = ((ts.hour_band == "pm 16-19") & (ts.weekend == 0) & (ts.pickup_borough == "Manhattan")).astype(int)
    slices["rush_manhattan"] = slice_table(ts, yte, final_te, "rush_manhattan", quote=quote_te).round(3)
    display(slices["rush_manhattan"])

## 6. Interpretation
### 6a · Linear coefficients as a price list (May only)
The distance-only linear model fitted on May, against the official TLC Rate #01 (\$3.00 start, \$0.70 per 1/5 mile = \$3.50/mi, or \$0.70/min below 12 mph; https://www.nyc.gov/site/tlc/passengers/taxi-fare.page, checked 2026-11-15).

In [ ]:
for fs in ["f0_distance"]:
    lin = make_pipeline(make_model("linear"), fs).fit(train, ytr).named_steps["model"]
    print(fs, "-> start charge ${:.2f}, per mile ${:.2f}".format(lin.intercept_, lin.coef_[0]))
print(f"TLC official: start ${TLC_START:.2f}, per mile ${TLC_PER_MILE:.2f}")
print("June MAE of the TLC rule:", june_tab.loc["2a rate rule (TLC $3 + $3.50/mi)", "mae"], "| bias:", june_tab.loc["2a rate rule (TLC $3 + $3.50/mi)", "bias"])

FILL_6A

### 6b · The time-charge story: residual vs hour and vs speed proxy (July)
Fares are billed for time in slow traffic; duration is unknown at booking. The speed band below uses duration **for analysis only** — it is not, and cannot be, a feature.

In [ ]:
if RUN_JULY:
    ts["res"] = yte - final_te
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ts.groupby("hour")["res"].mean().plot(ax=ax[0], marker="o", title="mean residual (actual - predicted) by hour, July"); ax[0].axhline(0, color="k", lw=.8)
    ts.groupby("pickup_borough")["res"].mean().plot(kind="bar", ax=ax[1], title="mean residual by pickup borough, July")
    plt.tight_layout()
    dur_h = (test.tpep_dropoff_datetime - test.tpep_pickup_datetime).dt.total_seconds() / 3600
    ts["speed_band"] = pd.cut(test.trip_distance / dur_h, [0, 6, 10, 15, 25, 100])
    slices["speed_band"] = slice_table(ts, yte, final_te, "speed_band", quote=quote_te).round(3)
    display(slices["speed_band"])
    save_results({"scored_at": LOCK.read_text(),
                  "model": FINAL_MODEL, "feature_set": FINAL_FEATURE_SET, "params": FINAL_PARAMS, "quote_rule": rule,
                  "july_metrics": july_tab.to_dict(orient="index"),
                  "july_vs_median_bootstrap": vs_median.to_dict(orient="index"),
                  "july_vs_rate_rule_bootstrap": vs_rule.to_dict(orient="index"),
                  "july_quote": july_quote, "july_slices": {k: v.reset_index().astype({v.index.name: str}).to_dict(orient="records") for k, v in slices.items()}},
                 "july_final.json")

### 6c · Aggregates for the project website (same single July run)
Saved now so that no later script ever has to score July again. Only aggregates (with n) leave the notebook.

In [ ]:
if RUN_JULY:
    a = pd.DataFrame({"y": yte, "p": final_te, "q": quote_te, "pu": test.PULocationID.to_numpy(), "do": test.DOLocationID.to_numpy(),
                      "hour": ts.hour.to_numpy(), "weekend": ts.weekend.to_numpy(), "dist": test.trip_distance.to_numpy()})
    a["res"] = a.y - a.p; a["abs"] = a.res.abs(); a["cov"] = (a.y <= a.q).astype(float)
    agg = dict(n=("y", "size"), mae=("abs", "mean"), bias=("res", "mean"), coverage=("cov", "mean"), mean_fare=("y", "mean"))
    def table(g): return g.agg(**agg).round(4).reset_index().to_dict(orient="records")
    edges_res = np.arange(-20, 30.5, 0.5); edges_fare = np.arange(0, 102, 2)
    h2, _, _ = np.histogram2d(np.clip(a.p, 0, 99.9), np.clip(a.y, 0, 99.9), bins=[edges_fare, edges_fare])
    dec = pd.qcut(a.p, 10, labels=False)
    analytics = {
        "n": len(a),
        "by_pickup_zone": table(a.groupby("pu")), "by_dropoff_zone": table(a.groupby("do")),
        "by_hour_weekend": table(a.groupby(["weekend", "hour"])),
        "residual_hist": {"edges": edges_res.tolist(), "counts": np.histogram(np.clip(a.res, -20, 29.99), edges_res)[0].tolist()},
        "pred_vs_actual": {"edges": edges_fare.tolist(), "counts": h2.astype(int).tolist()},
        "calibration_deciles": a.groupby(dec).agg(n=("y", "size"), mean_pred=("p", "mean"), mean_actual=("y", "mean"), median_actual=("y", "median")).round(4).to_dict(orient="records"),
    }
    save_results(analytics, "july_analytics.json")

## 7. Summary (copied to MODEL_CARD.md)
FILL_SUMMARY